# Persona 2 — Reparar los documentos que quedaron solo con resumen
Complemento del notebook `persona2_texto_glaciares_v2`. **No repite** los 366 documentos ya procesados.

**Qué pasó:** algunos PDFs (sobre todo informes antiguos de la ANA) son fotos de páginas con un **sello de texto** del repositorio en cada hoja. Como la página "tenía texto", no se trató como escaneada y no se le aplicó OCR; al quitar el sello repetido, el documento quedó vacío.

**Qué hace este notebook:**
1. Diagnostica cada documento en estado `solo_metadatos` (texto e imágenes por página, causa probable).
2. Lo reprocesa con una detección mejorada: si una imagen cubre la mayor parte de la hoja, la página se lee con **OCR** aunque tenga un sello.
3. Regenera **todas** las salidas (`chunks.jsonl`, `documentos_procesados.csv`, reporte…) con los 398 documentos juntos, leyendo lo ya procesado desde `json/`.

**Cómo correrlo:** en Colab (instala Tesseract con español sola), **CPU**, *Entorno de ejecución → Ejecutar todas*. Tiempo estimado: ~3–5 s por página con OCR (máx. 40 páginas por documento), aprox. 30–90 minutos para 32 documentos. Es reanudable: el estado se guarda documento por documento.

## 0. Carpetas

In [1]:
CARPETA_CORPUS_LOCAL = r"G:\Mi unidad\proyecto_gc\corpus_glaciares"
CARPETA_TEXTO_LOCAL  = r"G:\Mi unidad\proyecto_gc\texto_glaciares"

import os, sys
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/proyecto_gc"
    CARPETA_CORPUS, CARPETA_TEXTO = os.path.join(BASE, "corpus_glaciares"), os.path.join(BASE, "texto_glaciares")
else:
    CARPETA_CORPUS, CARPETA_TEXTO = CARPETA_CORPUS_LOCAL, CARPETA_TEXTO_LOCAL
CARPETA_PDFS = os.path.join(CARPETA_CORPUS, "pdfs")
print("Entrada:", CARPETA_CORPUS, "\nSalida: ", CARPETA_TEXTO)

Mounted at /content/drive
Entrada: /content/drive/MyDrive/proyecto_gc/corpus_glaciares 
Salida:  /content/drive/MyDrive/proyecto_gc/texto_glaciares


## 1. Configuración ✏️

In [2]:
CONFIG = {
    "CHUNK_OBJETIVO": 1500, "CHUNK_MAXIMO": 2200, "SOLAPE": 200,   # iguales al notebook principal
    "EXTRAER_TABLAS": True, "MAX_PAGINAS_TABLAS": 80,
    "MAX_PAGINAS_OCR": 40,      # páginas con OCR por documento (~3-5 s cada una)
    "ESTADOS_A_REPARAR": ["solo_metadatos"],
}
RUTA_TESSDATA = r"C:\Program Files\Tesseract-OCR\tessdata"   # solo Jupyter en Windows

## 2. Código
Solo se ejecuta.

In [3]:
%pip install -q -U pymupdf
import os, sys
if "google.colab" in sys.modules:
    os.system("apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null")

# -*- coding: utf-8 -*-
"""
Persona 2 — Del PDF al texto estructurado (caso Glaciares)
Proyecto: Enterprise Brain — Gestión del Conocimiento (Knowledge Graph + GraphRAG)

Entrada (Persona 1): papers.csv + pdfs/ + pdfs_manifest.csv  (398 documentos INAIGEM, ANA, SENAMHI)
Para cada PDF:
  1. Lee cada línea con su tipografía y posición (PyMuPDF).
  2. Detecta páginas escaneadas y, si Tesseract está instalado, les aplica OCR (español).
  3. Limpia: cabeceras y pies repetidos, números de página, índice / tabla de contenido,
     ligaduras y palabras cortadas con guion.
  4. Detecta secciones de informes técnicos en español (y de artículos en inglés) y las normaliza:
     resumen, introduccion, objetivos, antecedentes, area_estudio, metodologia, resultados,
     discusion, conclusiones, recomendaciones, referencias, anexos...
  5. Separa la primera página (institución, fecha), leyendas (Figura, Foto, Cuadro, Mapa) y tablas.
  6. Divide el texto en fragmentos con procedencia (documento, sección, páginas) y marca
     "pistas" del dominio (laguna, glaciar, peligro, poblado, obra, ubicación, medición)
     para que la Persona 3 priorice la extracción de entidades y relaciones.
Es reanudable: no reprocesa los PDFs que no cambiaron.
"""
import os, re, json, gzip, hashlib, unicodedata
from collections import Counter
from datetime import datetime, timezone

import pandas as pd

try:
    import pymupdf
except ImportError:  # versiones antiguas
    import fitz as pymupdf

VERSION = "2.0-glaciares"


def ahora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", t or "") if unicodedata.category(c) != "Mn")


def normalizar_texto(t):
    t = unicodedata.normalize("NFKC", t or "")  # ligaduras ﬁ -> fi, etc.
    t = t.replace("­", "").replace("​", "").replace("﻿", "")
    return re.sub(r"[ \t]+", " ", t).strip()


def nombre_archivo(paper_id):
    """Mismo nombre que usa la Persona 1 para el PDF (válido en Windows)."""
    return re.sub(r"[^\w.-]", "_", str(paper_id))


# ======================================================================
# 1. OCR (opcional, requiere Tesseract instalado)
# ======================================================================
def preparar_ocr(usar_ocr=True, ruta_tesseract=""):
    """Devuelve (disponible, idioma, mensaje)."""
    if not usar_ocr:
        return False, "", "OCR desactivado en la configuración"
    if ruta_tesseract:
        os.environ["TESSDATA_PREFIX"] = ruta_tesseract
    try:
        tessdata = os.environ.get("TESSDATA_PREFIX") or pymupdf.get_tessdata()
    except Exception:
        tessdata = ""
    if not tessdata or not os.path.isdir(tessdata):
        return False, "", ("Tesseract no está instalado: los PDFs escaneados usarán el resumen de los metadatos. "
                           "Ver la sección 0 para instalarlo.")
    os.environ["TESSDATA_PREFIX"] = tessdata
    idiomas = {f[:-12] for f in os.listdir(tessdata) if f.endswith(".traineddata")}
    if "spa" in idiomas:
        return True, "spa+eng" if "eng" in idiomas else "spa", f"OCR disponible (español) en {tessdata}"
    if "eng" in idiomas:
        return True, "eng", f"OCR disponible, pero sin el idioma español (spa.traineddata) en {tessdata}: se usará inglés"
    return False, "", f"Tesseract sin idiomas en {tessdata}"


# ======================================================================
# 2. Lectura de líneas con tipografía
# ======================================================================
def es_negrita(span):
    fuente = (span.get("font") or "").lower()
    return bool(span.get("flags", 0) & 16) or any(k in fuente for k in ("bold", "black", "heavy", "semibold", "negrita", ",b", "-b"))


def unir_spans(spans):
    """Une los tramos de una línea; si hay separación visual sin espacio, agrega uno."""
    out, x_prev = "", None
    for sp in spans:
        t = sp.get("text", "")
        if not t:
            continue
        x0 = sp["bbox"][0]
        if out and x_prev is not None and not out.endswith(" ") and not t.startswith(" ") \
                and x0 - x_prev > 0.15 * sp.get("size", 10):
            out += " "
        out += t
        x_prev = sp["bbox"][2]
    return out


def lineas_de_pagina(page, pno, textpage=None):
    lineas = []
    W, H = page.rect.width, page.rect.height
    d = page.get_text("dict", flags=pymupdf.TEXT_MEDIABOX_CLIP, textpage=textpage)
    for bi, b in enumerate(d.get("blocks", [])):
        if b.get("type") != 0:
            continue
        for li, ln in enumerate(b.get("lines", [])):
            dx, dy = ln.get("dir", (1, 0))
            if abs(dy) > 0.5:  # texto vertical en el margen
                continue
            spans = [s for s in ln.get("spans", []) if s.get("text", "").strip()]
            if not spans:
                continue
            texto = normalizar_texto(unir_spans(ln["spans"]))
            if not texto:
                continue
            principal = max(spans, key=lambda s: len(s["text"].strip()))
            x0, y0, x1, y1 = ln["bbox"]
            lineas.append({
                "page": pno, "block": bi, "line": li, "text": texto,
                "size": round(principal.get("size", 0), 1),
                "bold": sum(len(s["text"]) for s in spans if es_negrita(s)) >= 0.6 * sum(len(s["text"]) for s in spans),
                "x0": x0, "y0": y0, "x1": x1, "y1": y1, "W": W, "H": H,
            })
    return lineas


def cobertura_imagen(page):
    """Fracción de la página cubierta por imágenes (0 a 1)."""
    area = abs(page.rect)
    if not area:
        return 0.0
    total = 0.0
    try:
        for im in page.get_image_info():
            r = pymupdf.Rect(im["bbox"]) & page.rect
            total += abs(r)
    except Exception:
        return 0.0
    return min(1.0, total / area)


def leer_documento(doc, ocr=(False, ""), max_paginas_ocr=40, min_car_pagina=80, forzar_ocr=False):
    """Lee todas las páginas. Una página se trata como escaneada si casi no tiene texto y tiene
    imágenes, o si una imagen cubre la mayor parte de la hoja y el texto es poco (p. ej. un sello
    o una marca del repositorio sobre la foto de la página). Con OCR disponible se lee con Tesseract.
    forzar_ocr=True: se aplica OCR a toda página con imagen grande (para reparar documentos)."""
    ocr_ok, idioma = ocr
    lineas, escaneadas, con_ocr, errores_ocr = [], [], [], 0
    for pno, page in enumerate(doc):
        lp = lineas_de_pagina(page, pno)
        n_car = sum(len(l["text"]) for l in lp)
        cob = cobertura_imagen(page) if (n_car < 1500 or forzar_ocr) else 0.0
        escaneada = ((n_car < min_car_pagina and cob > 0) or (cob >= 0.6 and n_car < 600)
                     or (forzar_ocr and cob >= 0.3 and n_car < 1500))
        if escaneada:
            escaneadas.append(pno + 1)
            if ocr_ok and len(con_ocr) < max_paginas_ocr:
                try:
                    tp = page.get_textpage_ocr(language=idioma, dpi=200, full=True)
                    lp_ocr = lineas_de_pagina(page, pno, textpage=tp)
                    if sum(len(l["text"]) for l in lp_ocr) > n_car:
                        for l in lp_ocr:
                            l["ocr"] = True
                        lp = lp_ocr
                        con_ocr.append(pno + 1)
                except Exception:
                    errores_ocr += 1
        lineas += lp
    return lineas, {"paginas_escaneadas": escaneadas, "paginas_ocr": con_ocr, "errores_ocr": errores_ocr}


def quitar_cabeceras_pies(lineas, n_paginas, margen=0.09, min_frac=0.35):
    """Elimina líneas en los márgenes que se repiten en muchas páginas
    (p. ej. 'INAIGEM - Dirección de Investigación en Glaciares') y números de página."""
    def clave(l):
        return re.sub(r"\d+", "#", sin_tildes(l["text"].lower()))

    def en_margen(l):
        return l["y1"] < l["H"] * margen or l["y0"] > l["H"] * (1 - margen)

    conteo = {}
    for l in lineas:
        if en_margen(l):
            conteo.setdefault(clave(l), set()).add(l["page"])
    umbral = max(2, int(round(n_paginas * min_frac)))
    repetidas = {k for k, pags in conteo.items() if len(pags) >= umbral}
    salida, quitadas = [], 0
    for l in lineas:
        if en_margen(l) and (clave(l) in repetidas or
                             re.fullmatch(r"(p[aá]g(ina)?\.?\s*)?[\divxlc\-–— /de]{1,12}", l["text"].lower())):
            quitadas += 1
            continue
        salida.append(l)
    return salida, quitadas


# ======================================================================
# 3. Índice (tabla de contenido)
# ======================================================================
PAT_LINEA_INDICE = re.compile(r"[A-Za-zÁÉÍÓÚÑáéíóúñ].*?(\.{3,}|…{2,}|(\s\.){3,}|_{3,})\s*\d{1,3}\s*$")
PAT_FIN_NUMERO = re.compile(r"[A-Za-zÁÉÍÓÚÑáéíóúñ)]\s+\d{1,3}\s*$")


def paginas_indice(lineas, max_pagina=10):
    """Páginas (entre las primeras) donde la mayoría de líneas terminan en número de página."""
    por_pag = {}
    for l in lineas:
        if l["page"] < max_pagina:
            por_pag.setdefault(l["page"], []).append(l["text"])
    out = set()
    for p, textos in por_pag.items():
        n = sum(bool(PAT_LINEA_INDICE.search(t) or PAT_FIN_NUMERO.search(t)) for t in textos)
        cabecera = any(re.match(r"^(indice|contenido|tabla de contenido|indice general)\b", sin_tildes(t.lower()))
                       for t in textos[:6])
        if n >= 5 and (n >= 0.4 * len(textos) or cabecera):
            out.add(p)
    return out


# ======================================================================
# 4. Encabezados y tipos de sección (español + inglés)
# ======================================================================
PAT_NUMERADO = re.compile(
    r"^(?P<num>\d{1,2}(?:\.\d{1,2}){0,3}|[A-H](?:\.\d{1,2}){0,3}|[IVX]{1,5})[.)\-–]?\s+"
    r"(?P<titulo>[A-ZÁÉÍÓÚÑÜ¿(][^\n]{1,110})$")
PAT_SOLO_NUMERO = re.compile(r"^(\d{1,2}(?:\.\d{1,2}){0,3}|[IVX]{1,5})[.)]?$")
PAT_CAPITULO = re.compile(r"^cap[ií]tulo\s+(?P<num>[ivxlc\d]+)\b\.?\s*[:\-–.]?\s*(?P<titulo>.*)$", re.I)
PAT_LEYENDA = re.compile(
    r"^(figura|fig\.|foto(graf[ií]a)?|imagen|tabla|cuadro|gr[aá]fico|mapa|l[aá]mina|plano|esquema|"
    r"figure|table|photo)\s*(n\s*[°º.]\s*|nro\.?\s*|no\.?\s*)?\d+[a-z]?(\.\d+)?\s*[:.\-–|]", re.I)

NOMBRES_CONOCIDOS = {
    # español
    "resumen", "resumen ejecutivo", "presentacion", "prologo", "introduccion", "objetivo", "objetivos",
    "objetivo general", "objetivos especificos", "antecedentes", "marco teorico", "marco conceptual",
    "justificacion", "alcance", "alcances", "ubicacion", "ubicacion y acceso", "accesibilidad",
    "area de estudio", "zona de estudio", "ambito de estudio", "descripcion del area de estudio",
    "metodologia", "materiales y metodos", "metodos", "trabajos de campo", "trabajo de campo",
    "resultados", "resultados y discusion", "discusion", "analisis", "evaluacion", "diagnostico",
    "conclusiones", "conclusion", "conclusiones y recomendaciones", "recomendaciones",
    "referencias", "referencias bibliograficas", "bibliografia", "literatura citada",
    "anexos", "anexo", "apendice", "panel fotografico", "registro fotografico", "agradecimientos",
    "indice", "contenido", "tabla de contenido", "glosario", "abreviaturas", "siglas y acronimos",
    # inglés (artículos científicos del corpus)
    "abstract", "summary", "introduction", "background", "related work", "study area", "data and methods",
    "methods", "methodology", "materials and methods", "data", "results", "discussion", "conclusion",
    "conclusions", "acknowledgments", "acknowledgements", "references", "appendix", "supplementary material",
}

TIPOS_REGEX = [
    ("indice", r"^(indice|contenido|tabla de contenido|indice general|lista de (figuras|tablas|cuadros|fotos|fotografias|mapas|anexos)|"
               r"indice de (figuras|tablas|cuadros|fotografias|mapas|anexos)|glosario|abreviaturas|siglas)"),
    ("agradecimientos", r"agradecimiento|acknowledg"),
    ("referencias", r"^(referencias|bibliografia|literatura citada|fuentes consultadas|references|bibliography)"),
    ("anexos", r"^(anexo|apendice|appendi|panel fotografico|registro fotografico|supplementa)"),
    ("resumen", r"^(resumen|resumen ejecutivo|abstract|summary|sumario)$"),
    ("presentacion", r"^(presentacion|prologo|prefacio|foreword|preface)"),
    ("introduccion", r"^(introduc|generalidades)"),
    ("objetivos", r"objetivo|^finalidad|^alcances?$|^justificacion"),
    ("antecedentes", r"antecedente|marco teorico|marco conceptual|estado del arte|background|related work|literature"),
    ("area_estudio", r"ubicacion|area de estudio|zona de estudio|ambito|accesibilidad|localizacion|study area|"
                     r"descripcion (general )?(de la|del) (zona|area|laguna|glaciar|cuenca|subcuenca|nevado)|"
                     r"caracteristicas (generales|fisicas|de la laguna|del glaciar|de la cuenca)|geografia|clima de la zona"),
    ("conclusiones", r"^conclusi|conclusion"),
    ("recomendaciones", r"recomendaci"),
    ("discusion", r"^discusi|discussion"),
    ("metodologia", r"metodolog|metodo|materiales y|procedimiento|trabajos? de campo|methods?\b|methodology|"
                    r"^data\b|datos y|levantamiento|instrumentacion|equipos? utilizados"),
    ("resultados", r"resultado|analisis|evaluacion|diagnostico|caracterizacion|inspeccion|monitoreo|"
                   r"inventario|batimetri|peligro|riesgo|results|evaluation|analysis"),
]
TIPOS_SIN_FRAGMENTOS = ("indice", "referencias", "agradecimientos", "front_matter")


def tipo_de_titulo(titulo):
    t = re.sub(r"[^a-z ]", " ", sin_tildes(titulo.lower()))
    t = re.sub(r"\s+", " ", t).strip()
    for tipo, pat in TIPOS_REGEX:
        if re.search(pat, t):
            return tipo
    return "otro"


def limpiar_titulo(t):
    return re.sub(r"\s+", " ", t).strip(" .:-–")


def detectar_encabezado(l, cuerpo):
    """Devuelve (numero, titulo, nivel) si la línea es un encabezado; si no, None."""
    texto = l["text"].strip()
    if len(texto) > 120 or len(texto.split()) > 16 or PAT_LEYENDA.match(texto):
        return None
    letras = re.sub(r"[^A-Za-zÁÉÍÓÚÑÜáéíóúñü]", "", texto)
    mayusculas = len(letras) >= 4 and letras.isupper()
    destacado = l["bold"] or l["size"] >= cuerpo + 0.8 or mayusculas
    m = PAT_CAPITULO.match(texto)
    if m and (destacado or len(texto.split()) <= 6):
        return m.group("num").upper(), limpiar_titulo(m.group("titulo")) or texto, 1
    sin_num = limpiar_titulo(re.sub(r"^(\d{1,2}(?:\.\d{1,2}){0,3}|[A-H]|[IVX]{1,5})[.)\-–]?\s+", "", texto))
    if sin_tildes(sin_num.lower()) in NOMBRES_CONOCIDOS and (destacado or l["size"] >= cuerpo - 0.2):
        num = texto[: len(texto) - len(sin_num)].strip(" .)-–") if texto.lower() != sin_num.lower() else ""
        return num, sin_num, (num.count(".") + 1) if num else 1
    m = PAT_NUMERADO.match(texto)
    if m and destacado:
        num, titulo = m.group("num"), limpiar_titulo(m.group("titulo"))
        if re.match(r"^[A-H]$", num) and not (l["bold"] or mayusculas):
            return None
        if titulo.endswith(".") and len(titulo.split()) > 7:
            return None  # oración numerada (lista de conclusiones o recomendaciones)
        if sum(c.isdigit() for c in titulo) > 6:
            return None
        nivel = num.count(".") + 1 if num[0].isdigit() else 1
        return num, titulo, nivel
    return None


# ======================================================================
# 5. Procesamiento de un PDF
# ======================================================================
PAT_INSTITUCION = re.compile(
    r"\b(INAIGEM|ANA|SENAMHI|INDECI|CENEPRED|INGEMMET|IGP|INRENA|ELECTROPER[UÚ]|HIDRANDINA|EGENOR|"
    r"autoridad nacional del agua|instituto\s+\w+|servicio nacional|ministerio|direcci[oó]n|subdirecci[oó]n|"
    r"unidad de glaciolog\w*|gobierno regional|municipalidad|universi\w+|facultad|"
    r"university|institute|department|laborator\w+|centro de)\b", re.I)
PAT_EMAIL = re.compile(r"[\w.+-]+@[\w-]+(\.[\w-]+)+")
PAT_FECHA = re.compile(r"\b(enero|febrero|marzo|abril|mayo|junio|julio|agosto|se(p)?tiembre|octubre|noviembre|diciembre)"
                       r"\s*(de|del)?\s*(19|20)\d{2}\b", re.I)

PISTAS = {
    "laguna": r"\blagunas?\b|\blake\b|\blakes\b",
    "glaciar": r"\bglaciar\w*|\bnevados?\b|\bglacier\w*|\bcobertura glaciar|\bmasa de hielo",
    "peligro": r"riesgo|peligr\w*|aluvi[oó]n\w*|desborde|glof|avalanch\w*|huaico|huayco|deslizamiento|vulnerab\w*|hazard|outburst",
    "poblado": r"centros? poblados?|ciudad de|distrito de|localidad|caser[ií]o|comunidad campesina|poblaci[oó]n|habitantes|viviendas",
    "obra": r"obras? de (seguridad|control)|\bdique\w*|t[uú]nel\w*|canal de (desag[uü]e|descarga)|sif[oó]n\w*|aliviadero|"
            r"represamiento|presa de|desembalse|sistema de alerta temprana",
    "ubicacion": r"cordillera|subcuenca|microcuenca|cuenca del?|departamento|provincia de|regi[oó]n (de )?[A-ZÁÉÍÓÚ]|quebrada",
    "medicion": r"\d+(?:[.,]\d+)?\s?(?:m\.?\s?s\.?\s?n\.?\s?m\.?|msnm|km2|km²|m2|m²|m3|m³|hm3|hm³|mmc|ha\b|%|m/a|m/año)",
}
PISTAS_RE = {k: re.compile(v, re.I) for k, v in PISTAS.items()}
STOP_ES = set("de la que el en los del se las por un para con una su al lo como más pero sus le ya o fue este ha sí porque esta son entre "
              "cuando muy sin sobre también me hasta hay donde quien desde todo nos durante todos uno les ni contra otros ese eso".split())
STOP_EN = set("the of and to in is that for it as with was on be by this are from at or an which have has were not but their these".split())


def idioma(texto):
    pal = re.findall(r"[a-záéíóúñ]+", texto[:20000].lower())
    es, en = sum(p in STOP_ES for p in pal), sum(p in STOP_EN for p in pal)
    if es + en < 20:
        return "desconocido"
    return "es" if es >= en else "en"


def unir_lineas(textos):
    """Une líneas de un párrafo reparando palabras cortadas con guion."""
    out = ""
    for t in textos:
        if not out:
            out = t
        elif out.endswith("-") and t[:1].islower() and not re.search(r"-\w+-$", out):
            out = out[:-1] + t
        else:
            out += " " + t
    return out.strip()


def procesar_pdf(ruta, paper_id, titulo_meta="", abstract_meta="", extraer_tablas=True,
                 max_paginas_tablas=80, ocr=(False, ""), max_paginas_ocr=40, forzar_ocr=False):
    res = {"paper_id": paper_id, "version": VERSION, "title": titulo_meta, "flags": [], "procesado": ahora()}
    try:
        doc = pymupdf.open(ruta)
    except Exception as e:
        res["flags"].append("pdf_no_abre")
        res["error"] = str(e)[:300]
        return res
    res["n_pages"] = doc.page_count

    lineas, info = leer_documento(doc, ocr, max_paginas_ocr, forzar_ocr=forzar_ocr)
    res.update(info)
    res["ocr_aplicado"] = bool(info["paginas_ocr"])
    res["ocr_disponible"] = bool(ocr[0])
    n_esc = len(info["paginas_escaneadas"])
    if n_esc:
        res["flags"].append(f"escaneado ({n_esc}/{doc.page_count} pág.)" if n_esc >= 0.5 * doc.page_count
                            else f"paginas_escaneadas ({n_esc})")
    if n_esc and not info["paginas_ocr"]:
        res["flags"].append("sin_ocr")

    # Tablas (antes del texto para no mezclar sus celdas con los párrafos). No en páginas OCR.
    tablas, zonas_tabla = [], {}
    if extraer_tablas:
        for pno in range(min(doc.page_count, max_paginas_tablas)):
            if pno + 1 in info["paginas_escaneadas"]:
                continue
            try:
                for t in doc[pno].find_tables().tables:
                    md = t.to_markdown(clean=True) if hasattr(t, "to_markdown") else ""
                    if md and md.count("\n") >= 2:
                        tablas.append({"page": pno + 1, "bbox": [round(v, 1) for v in t.bbox], "markdown": md})
                        zonas_tabla.setdefault(pno, []).append(t.bbox)
            except Exception:
                pass

    texto_total = " ".join(l["text"] for l in lineas)
    res["pdf_text_chars"] = len(texto_total)
    if texto_total.count("(cid:") > 20 or texto_total.count("�") > 50:
        res["flags"].append("texto_con_errores_de_codificacion")
    res["idioma"] = idioma(texto_total)

    lineas, n_quitadas = quitar_cabeceras_pies(lineas, doc.page_count)
    res["lineas_cabecera_pie_quitadas"] = n_quitadas
    pags_indice = paginas_indice(lineas)
    res["paginas_indice"] = sorted(p + 1 for p in pags_indice)

    def en_tabla(l):
        for (x0, y0, x1, y1) in zonas_tabla.get(l["page"], []):
            if l["x0"] >= x0 - 2 and l["x1"] <= x1 + 2 and l["y0"] >= y0 - 2 and l["y1"] <= y1 + 2:
                return True
        return False

    tamanos = Counter()
    for l in lineas:
        tamanos[l["size"]] += len(l["text"])
    cuerpo = tamanos.most_common(1)[0][0] if tamanos else 10.0
    res["font_size_cuerpo"] = cuerpo

    # Título: línea(s) con la mayor tipografía en la primera página (portada)
    p1 = [l for l in lineas if l["page"] == 0]
    titulo_ids = set()
    if p1:
        mayor = max(l["size"] for l in p1)
        if not titulo_meta:
            res["title"] = " ".join(l["text"] for l in p1 if l["size"] == mayor)[:300]
        if mayor >= cuerpo + 2:
            titulo_ids = {(l["page"], l["block"], l["line"]) for l in p1 if l["size"] == mayor}

    secciones = [{"heading": "PORTADA", "num": "", "level": 0, "type": "front_matter", "parrafos": []}]
    leyendas, lineas_front, n_lineas_indice = [], [], 0
    buffer, buf_key, buf_pag = [], None, None
    pendiente_num = None

    def cerrar_parrafo():
        nonlocal buffer, buf_key, buf_pag
        if buffer:
            texto = unir_lineas(buffer)
            if PAT_LEYENDA.match(texto) and len(texto) < 400:
                leyendas.append({"page": buf_pag + 1, "text": texto, "section": secciones[-1]["heading"]})
            elif len(texto) > 1:
                previos = secciones[-1]["parrafos"]
                if previos and previos[-1]["text"].endswith("-") and texto[:1].islower():
                    previos[-1]["text"] = previos[-1]["text"][:-1] + texto   # palabra cortada entre bloques
                else:
                    previos.append({"page": buf_pag + 1, "text": texto})
        buffer, buf_key, buf_pag = [], None, None

    for l in lineas:
        if (l["page"], l["block"], l["line"]) in titulo_ids or en_tabla(l):
            continue
        if l["page"] in pags_indice or PAT_LINEA_INDICE.search(l["text"]):
            n_lineas_indice += 1
            continue
        if pendiente_num is not None:
            l = dict(l, text=f"{pendiente_num} {l['text']}")
            pendiente_num = None
        if PAT_SOLO_NUMERO.match(l["text"]) and (l["bold"] or l["size"] >= cuerpo + 0.8):
            cerrar_parrafo()
            pendiente_num = l["text"].strip(" .)")
            continue
        enc = detectar_encabezado(l, cuerpo)
        if enc:
            cerrar_parrafo()
            num, titulo, nivel = enc
            secciones.append({"heading": titulo, "num": num, "level": nivel, "type": tipo_de_titulo(titulo),
                              "page_start": l["page"] + 1, "parrafos": []})
            continue
        if len(secciones) == 1:
            lineas_front.append(l["text"])
        clave = (l["page"], l["block"])
        if clave != buf_key:
            cerrar_parrafo()
            buf_key, buf_pag = clave, l["page"]
        buffer.append(l["text"])
    cerrar_parrafo()
    res["lineas_indice_quitadas"] = n_lineas_indice

    # Tipos: herencia en subsecciones y deducción por posición
    padre = None
    for s in secciones[1:]:
        if s["level"] <= 1:
            padre = s
        elif s["type"] == "otro" and padre is not None:
            s["type"] = padre["type"]
    i_intro = next((i for i, s in enumerate(secciones) if s["type"] in ("introduccion", "objetivos", "antecedentes")), None)
    i_fin = next((i for i, s in enumerate(secciones) if s["type"] in ("conclusiones", "recomendaciones", "referencias")), None)
    i_ref = next((i for i, s in enumerate(secciones) if s["type"] == "referencias"), None)
    for i, s in enumerate(secciones):
        if s["type"] != "otro":
            continue
        if i_ref is not None and i > i_ref:
            s["type"] = "anexos"
        elif i_intro is not None and i > i_intro and (i_fin is None or i < i_fin):
            s["type"] = "desarrollo"   # capítulo propio del informe (p. ej. "Descripción de peligros")
    if i_ref is not None:
        for s in secciones[i_ref + 1:]:
            if s["type"] not in ("agradecimientos", "referencias", "anexos", "conclusiones", "recomendaciones"):
                s["tipo_original"] = s["type"]
                s["type"] = "anexos"

    # Resumen: sección del PDF o, si no hay, el de los metadatos (lo normal en informes técnicos)
    resumen_sec = next((s for s in secciones if s["type"] == "resumen"), None)
    resumen = " ".join(p["text"] for p in resumen_sec["parrafos"]) if resumen_sec else (abstract_meta or "")
    res["resumen_fuente"] = "pdf" if resumen_sec else ("metadatos" if abstract_meta else "no_encontrado")

    front_texto = "\n".join(p["text"] for p in secciones[0]["parrafos"])
    p1_texto = "\n".join(l["text"] for l in lineas if l["page"] == 0)
    candidatas = []
    for ln in lineas_front[:60] + p1_texto.split("\n")[:60]:
        if PAT_INSTITUCION.search(ln) and not PAT_EMAIL.search(ln) and 3 < len(ln) < 200:
            candidatas.append(ln.strip(" ,"))
    fechas = [m.group(0) for m in PAT_FECHA.finditer(front_texto + "\n" + p1_texto)]
    res.update({
        "resumen": resumen, "front_matter": front_texto[:5000],
        "instituciones_candidatas": list(dict.fromkeys(candidatas))[:20],
        "fechas_portada": list(dict.fromkeys(fechas))[:5],
        "emails": list(dict.fromkeys(m.group(0) for m in PAT_EMAIL.finditer(front_texto)))[:20],
        "first_page_text": p1_texto[:6000],
        "captions": leyendas, "tables": tablas,
    })

    salida = []
    for s in secciones:
        texto = "\n\n".join(p["text"] for p in s["parrafos"])
        if s["type"] == "front_matter" and not texto:
            continue
        paginas = [p["page"] for p in s["parrafos"]] or [s.get("page_start", 1)]
        salida.append({"idx": len(salida), "heading": s["heading"], "num": s["num"], "level": s["level"],
                       "type": s["type"], "page_start": min(paginas), "page_end": max(paginas),
                       "n_chars": len(texto), "parrafos": s["parrafos"], "text": texto})
    res["sections"] = salida

    tipos = {s["type"] for s in salida}
    principales = [s for s in salida if s["level"] == 1 and s["type"] != "front_matter"]
    sin_estructura = len(principales) < 3
    excluir = ("indice", "referencias", "agradecimientos") if sin_estructura else TIPOS_SIN_FRAGMENTOS
    n_util = sum(s["n_chars"] for s in salida if s["type"] not in excluir)
    if n_util < 1500:
        res["flags"].append("sin_texto_util")
    elif sin_estructura:
        res["flags"].append("estructura_no_detectada")
    res["tipos_detectados"] = sorted(tipos)
    res["n_sections"] = len(salida)
    res["n_chars_texto"] = sum(s["n_chars"] for s in salida)
    res["n_chars_util"] = n_util
    doc.close()
    return res


# ======================================================================
# 6. Fragmentación con procedencia y pistas del dominio
# ======================================================================
def dividir_oraciones(texto):
    return [o for o in re.split(r"(?<=[.!?;])\s+(?=[A-ZÁÉÍÓÚÑ¿¡(\[\d])", texto) if o.strip()]


def pistas(texto):
    return [k for k, p in PISTAS_RE.items() if p.search(texto)]


def fragmentar(res, meta=None, objetivo=1500, maximo=2200, solape=200):
    """Fragmentos por sección respetando párrafos. Sin estructura: todo el cuerpo.
    Sin texto útil (escaneado sin OCR): un único fragmento con título + resumen de los metadatos."""
    meta = meta or {}
    pid = res["paper_id"]
    base = {"paper_id": pid, "fuente": meta.get("fuente", ""), "title": meta.get("title", res.get("title", "")),
            "year": meta.get("year", "")}
    chunks = []
    secciones = res.get("sections", [])
    flags = res.get("flags", [])
    if "sin_texto_util" in flags or not secciones:
        texto = (base["title"] + ". " + (res.get("resumen") or meta.get("abstract", "") or "")).strip()
        if len(texto) > 20:
            p = pistas(texto)
            chunks.append({**base, "chunk_id": f"{pid}::meta::c000", "section_idx": -1, "section_type": "resumen",
                           "heading": "Resumen (metadatos)", "page_start": None, "page_end": None,
                           "text": texto, "n_chars": len(texto), "pistas": p, "prioridad": len(p),
                           "fuente_texto": "metadatos"})
        return chunks
    if "estructura_no_detectada" in flags:
        todos = [p for s in secciones for p in s["parrafos"] if s["type"] not in ("indice", "referencias", "agradecimientos")]
        secciones = [{"idx": 0, "heading": "CUERPO", "type": "cuerpo", "parrafos": todos}]
    paginas_ocr = set(res.get("paginas_ocr", []))
    for s in secciones:
        if s["type"] in TIPOS_SIN_FRAGMENTOS:
            continue
        piezas = []
        for p in s["parrafos"]:
            if len(p["text"]) <= maximo:
                piezas.append(p)
            else:
                actual = ""
                for o in dividir_oraciones(p["text"]):
                    while len(o) > maximo:   # oración enorme (texto OCR sin puntos)
                        piezas.append({"page": p["page"], "text": o[:objetivo]})
                        o = o[objetivo:]
                    if actual and len(actual) + len(o) + 1 > objetivo:
                        piezas.append({"page": p["page"], "text": actual})
                        actual = o
                    else:
                        actual = (actual + " " + o).strip()
                if actual:
                    piezas.append({"page": p["page"], "text": actual})
        texto, paginas, k = "", [], 0

        def emitir():
            nonlocal k
            t = texto.strip()
            if len(t) >= 40:
                p = pistas(t)
                chunks.append({
                    **base, "chunk_id": f"{pid}::s{s['idx']:02d}::c{k:03d}", "section_idx": s["idx"],
                    "section_type": s["type"], "heading": s["heading"],
                    "page_start": min(paginas), "page_end": max(paginas), "text": t, "n_chars": len(t),
                    "pistas": p, "prioridad": len([x for x in p if x != "medicion"]) + ("medicion" in p) * 0.5,
                    "fuente_texto": "ocr" if set(paginas) & paginas_ocr else "pdf",
                })
                k += 1

        for p in piezas:
            if texto and len(texto) + len(p["text"]) + 2 > objetivo:
                emitir()
                arrastre = ""
                for o in reversed(dividir_oraciones(texto)):
                    if len(arrastre) + len(o) > solape:
                        break
                    arrastre = (o + " " + arrastre).strip()
                texto, paginas = arrastre, paginas[-1:]
            texto = (texto + "\n\n" + p["text"]).strip()
            paginas.append(p["page"])
        emitir()
    return chunks


# ======================================================================
# 7. Proceso por lotes (reanudable)
# ======================================================================
def ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1=None):
    if manifiesto_p1 is not None and pid in manifiesto_p1.index:
        archivo = manifiesto_p1.loc[pid, "archivo"]
        if isinstance(archivo, str) and archivo:
            return os.path.join(carpeta_pdfs, archivo)
    return os.path.join(carpeta_pdfs, nombre_archivo(pid) + ".pdf")


def fila_manifiesto(pid, res, sha, reutilizado=False):
    flags = res.get("flags", [])
    estado = ("error" if "pdf_no_abre" in flags else
              "solo_metadatos" if "sin_texto_util" in flags else
              "ok_ocr" if res.get("ocr_aplicado") else "ok")
    return {"paper_id": pid, "estado": estado, "sha256_pdf": sha, "version": VERSION,
            "n_pages": res.get("n_pages"), "idioma": res.get("idioma", ""),
            "paginas_escaneadas": len(res.get("paginas_escaneadas", [])),
            "paginas_ocr": len(res.get("paginas_ocr", [])),
            "n_sections": res.get("n_sections", 0), "n_chars_util": res.get("n_chars_util", 0),
            "resumen_fuente": res.get("resumen_fuente", ""),
            "n_tablas": len(res.get("tables", [])), "n_leyendas": len(res.get("captions", [])),
            "tipos_detectados": ";".join(res.get("tipos_detectados", [])),
            "flags": ";".join(flags), "procesado": res.get("procesado", ahora()),
            "reutilizado": reutilizado}


def procesar_corpus(papers, carpeta_pdfs, carpeta_salida, manifiesto_p1=None, extraer_tablas=True,
                    max_paginas_tablas=80, ocr=(False, ""), max_paginas_ocr=40, forzar=False, log=print):
    carpeta_json = os.path.join(carpeta_salida, "json")
    os.makedirs(carpeta_json, exist_ok=True)
    ruta_man = os.path.join(carpeta_salida, "texto_manifest.csv")
    previo = (pd.read_csv(ruta_man, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
              if os.path.exists(ruta_man) else None)
    filas, resultados = [], {}
    total = len(papers)
    for i, f in enumerate(papers.itertuples(), start=1):
        pid = str(f.paper_id)
        ruta_pdf = ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1)
        ruta_json = os.path.join(carpeta_json, nombre_archivo(pid) + ".json")
        if not os.path.exists(ruta_pdf):
            filas.append({"paper_id": pid, "estado": "sin_pdf", "flags": "sin_pdf", "procesado": ahora()})
            log(f"   ❌ {pid}: no está el PDF ({os.path.basename(ruta_pdf)})")
            continue
        sha = sha256_archivo(ruta_pdf)
        reutilizar = (not forzar and previo is not None and pid in previo.index and os.path.exists(ruta_json)
                      and previo.loc[pid, "sha256_pdf"] == sha and str(previo.loc[pid, "version"]) == VERSION)
        res = None
        if reutilizar:
            with open(ruta_json, encoding="utf-8") as fh:
                res = json.load(fh)
            # si antes no había OCR y ahora sí, se reprocesan los escaneados
            if ocr[0] and res.get("paginas_escaneadas") and not res.get("ocr_disponible"):
                reutilizar, res = False, None
        if res is None:
            res = procesar_pdf(ruta_pdf, pid, getattr(f, "title", "") or "", getattr(f, "abstract", "") or "",
                               extraer_tablas, max_paginas_tablas, ocr, max_paginas_ocr)
            res["pdf_sha256"] = sha
            with open(ruta_json, "w", encoding="utf-8") as fh:
                json.dump(res, fh, ensure_ascii=False)
        resultados[pid] = res
        filas.append(fila_manifiesto(pid, res, sha, reutilizar))
        if i % 10 == 0 or i == total:
            log(f"   {i} / {total} procesados")
            pd.DataFrame(filas).to_csv(ruta_man, index=False, encoding="utf-8")
    manifiesto = pd.DataFrame(filas)
    manifiesto.to_csv(ruta_man, index=False, encoding="utf-8")
    return resultados, manifiesto


def cargar_resultados(papers, carpeta_salida):
    """Lee los resultados ya guardados en json/ (no reprocesa nada)."""
    carpeta_json = os.path.join(carpeta_salida, "json")
    resultados = {}
    for pid in papers.paper_id.astype(str):
        ruta = os.path.join(carpeta_json, nombre_archivo(pid) + ".json")
        if os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as fh:
                resultados[pid] = json.load(fh)
    return resultados


def diagnosticar_pdf(ruta, max_paginas=15):
    """Por qué un PDF quedó sin texto: caracteres e imágenes por página."""
    try:
        doc = pymupdf.open(ruta)
    except Exception as e:
        return {"error": str(e)[:200]}
    car, cob, muestra = [], [], ""
    for page in list(doc)[:max_paginas]:
        t = page.get_text()
        car.append(len(t.strip()))
        cob.append(cobertura_imagen(page))
        if not muestra and len(t.strip()) > 5:
            muestra = re.sub(r"\s+", " ", t.strip())[:120]
    n = doc.page_count
    doc.close()
    media_cob = sum(cob) / max(len(cob), 1)
    media_car = sum(car) / max(len(car), 1)
    if media_cob >= 0.5 and media_car < 1500:
        causa = "escaneado con poco texto (sello o marca) → OCR"
    elif media_cob < 0.05 and media_car < 80:
        causa = "sin texto ni imágenes (¿vectorial o dañado?)"
    elif media_cob < 0.5 and media_car < 300:
        causa = "mapa, plano o lámina con poco texto"
    else:
        causa = "revisar a mano"
    return {"paginas": n, "car_por_pagina": round(media_car), "cobertura_imagen": round(media_cob, 2),
            "texto_muestra": muestra, "causa_probable": causa}


def reprocesar_documentos(ids, papers, carpeta_pdfs, carpeta_salida, manifiesto_p1=None, ocr=(False, ""),
                          max_paginas_ocr=40, extraer_tablas=True, max_paginas_tablas=80, log=print):
    """Reprocesa solo los documentos indicados con OCR forzado y actualiza json/ y texto_manifest.csv."""
    meta = papers.set_index("paper_id")
    carpeta_json = os.path.join(carpeta_salida, "json")
    ruta_man = os.path.join(carpeta_salida, "texto_manifest.csv")
    man = pd.read_csv(ruta_man, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
    cambios = []
    for i, pid in enumerate(ids, start=1):
        ruta_pdf = ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1)
        if not os.path.exists(ruta_pdf):
            log(f"   ❌ {pid}: no está el PDF")
            continue
        antes = man.loc[pid, "estado"] if pid in man.index else ""
        m = meta.loc[pid] if pid in meta.index else {}
        titulo = m.get("title", "") if len(m) else ""
        resumen = m.get("abstract", "") if len(m) else ""
        res = procesar_pdf(ruta_pdf, pid, titulo if isinstance(titulo, str) else "",
                           resumen if isinstance(resumen, str) else "", extraer_tablas, max_paginas_tablas,
                           ocr, max_paginas_ocr, forzar_ocr=True)
        res["pdf_sha256"] = sha256_archivo(ruta_pdf)
        res["reparado"] = ahora()
        with open(os.path.join(carpeta_json, nombre_archivo(pid) + ".json"), "w", encoding="utf-8") as fh:
            json.dump(res, fh, ensure_ascii=False)
        fila = fila_manifiesto(pid, res, res["pdf_sha256"])
        for k, v in fila.items():
            if k != "paper_id":
                man.loc[pid, k] = v
        cambios.append({"paper_id": pid, "antes": antes, "ahora": fila["estado"],
                        "paginas_ocr": fila["paginas_ocr"], "car_utiles": fila["n_chars_util"]})
        log(f"   {i}/{len(ids)} {pid}: {antes} → {fila['estado']} ({fila['paginas_ocr']} pág. OCR, "
            f"{fila['n_chars_util']:,} car.)")
        man.reset_index().to_csv(ruta_man, index=False, encoding="utf-8")
    return pd.DataFrame(cambios), man.reset_index()


# ======================================================================
# 8. Salidas, reporte y documentación
# ======================================================================
def _jsonl(ruta, filas, comprimir=False):
    abrir = gzip.open if comprimir else open
    with abrir(ruta, "wt", encoding="utf-8") as fh:
        for x in filas:
            fh.write(json.dumps(x, ensure_ascii=False, default=str) + "\n")


def construir_salidas(resultados, papers, carpeta_salida, objetivo=1500, maximo=2200, solape=200):
    meta = papers.set_index("paper_id")
    carpeta_txt = os.path.join(carpeta_salida, "textos")
    os.makedirs(carpeta_txt, exist_ok=True)
    secciones, chunks, primeras, leyendas, tablas, docs = [], [], [], [], [], []
    for pid, r in resultados.items():
        m = meta.loc[pid].to_dict() if pid in meta.index else {}
        m = {k: ("" if (isinstance(v, float) and pd.isna(v)) else v) for k, v in m.items()}
        for s in r.get("sections", []):
            secciones.append({"paper_id": pid, "section_idx": s["idx"], "heading": s["heading"], "num": s["num"],
                              "level": s["level"], "section_type": s["type"], "page_start": s["page_start"],
                              "page_end": s["page_end"], "n_chars": s["n_chars"], "text": s["text"]})
        ch = fragmentar(r, m, objetivo, maximo, solape)
        chunks += ch
        # Texto limpio completo (sin índice ni referencias), para leer y para el RAG
        cuerpo = "\n\n".join(f"## {s['heading']}\n\n{s['text']}" for s in r.get("sections", [])
                             if s["type"] not in ("indice", "referencias") and s["text"])
        if not cuerpo:
            cuerpo = "\n\n".join(c["text"] for c in ch)
        with open(os.path.join(carpeta_txt, nombre_archivo(pid) + ".txt"), "w", encoding="utf-8") as fh:
            fh.write(f"# {m.get('title', r.get('title', ''))}\n\nFuente: {m.get('fuente', '')} · Año: {m.get('year', '')} · "
                     f"paper_id: {pid}\n\n{cuerpo}\n")
        primeras.append({"paper_id": pid, "title": m.get("title", ""), "fuente": m.get("fuente", ""),
                         "instituciones_candidatas": r.get("instituciones_candidatas", []),
                         "fechas_portada": r.get("fechas_portada", []), "emails": r.get("emails", []),
                         "first_page_text": r.get("first_page_text", "")})
        for c in r.get("captions", []):
            leyendas.append({"paper_id": pid, **c, "tipo": sin_tildes(c["text"].split()[0].lower().rstrip(".:"))})
        for k, t in enumerate(r.get("tables", [])):
            tablas.append({"table_id": f"{pid}::t{k:02d}", "paper_id": pid, **t})
        docs.append({"paper_id": pid, "title": m.get("title", ""), "fuente": m.get("fuente", ""), "year": m.get("year", ""),
                     "eje_principal": m.get("eje_principal", ""), "idioma": r.get("idioma", ""),
                     "n_pages": r.get("n_pages"), "n_chars_util": r.get("n_chars_util", 0),
                     "fuente_texto": ("metadatos" if ch and ch[0]["fuente_texto"] == "metadatos"
                                      else "pdf+ocr" if r.get("ocr_aplicado") else "pdf"),
                     "n_secciones": r.get("n_sections", 0), "n_fragmentos": len(ch),
                     "n_tablas": len(r.get("tables", [])), "resumen_fuente": r.get("resumen_fuente", ""),
                     "secciones": ";".join(r.get("tipos_detectados", [])), "observaciones": ";".join(r.get("flags", []))})
    df_sec = pd.DataFrame(secciones)
    df_chunks = pd.DataFrame(chunks)
    df_docs = pd.DataFrame(docs)
    df_sec.to_csv(os.path.join(carpeta_salida, "sections.csv"), index=False, encoding="utf-8")
    df_docs.to_csv(os.path.join(carpeta_salida, "documentos_procesados.csv"), index=False, encoding="utf-8")
    pd.DataFrame(leyendas).to_csv(os.path.join(carpeta_salida, "captions.csv"), index=False, encoding="utf-8")
    _jsonl(os.path.join(carpeta_salida, "chunks.jsonl"), chunks)
    _jsonl(os.path.join(carpeta_salida, "chunks.jsonl.gz"), chunks, comprimir=True)   # para GitHub (pesa menos)
    _jsonl(os.path.join(carpeta_salida, "first_pages.jsonl"), primeras)
    _jsonl(os.path.join(carpeta_salida, "tables.jsonl"), tablas)
    return df_docs, df_sec, df_chunks, pd.DataFrame(primeras), pd.DataFrame(leyendas), pd.DataFrame(tablas)


TIPOS_CLAVE = ["resumen", "introduccion", "objetivos", "area_estudio", "metodologia", "resultados", "conclusiones",
               "recomendaciones", "referencias"]


def reporte_extraccion(manifiesto, df_docs, df_sec, df_chunks, leyendas, tablas, n_docs, ocr_msg=""):
    ok, warn, err = "✅", "⚠️", "❌"
    chk = []
    estados = manifiesto["estado"].value_counts().to_dict()
    con_texto = estados.get("ok", 0) + estados.get("ok_ocr", 0)
    chk.append((ok if con_texto >= 0.9 * n_docs else warn,
                f"Documentos con texto del PDF: {con_texto}/{n_docs} (de ellos con OCR: {estados.get('ok_ocr', 0)})"))
    solo_meta = estados.get("solo_metadatos", 0)
    chk.append((ok if solo_meta == 0 else warn,
                f"Documentos solo con título + resumen (escaneados sin OCR): {solo_meta}"))
    sin_pdf = estados.get("sin_pdf", 0) + estados.get("error", 0)
    chk.append((ok if sin_pdf == 0 else err, f"Documentos sin PDF o con PDF dañado: {sin_pdf}"))
    sin_frag = n_docs - (df_chunks.paper_id.nunique() if len(df_chunks) else 0)
    chk.append((ok if sin_frag == 0 else err, f"Documentos sin ningún fragmento: {sin_frag}"))
    flags = manifiesto["flags"].fillna("")
    estructura = flags.str.contains("estructura_no_detectada").sum()
    chk.append((ok if estructura <= n_docs * 0.25 else warn,
                f"Documentos sin secciones detectadas (se fragmenta el cuerpo completo): {estructura}"))
    cobertura = {}
    for t in TIPOS_CLAVE:
        cobertura[t] = int(df_sec[df_sec.section_type == t].paper_id.nunique()) if len(df_sec) else 0
    if len(df_chunks):
        prior = (df_chunks.prioridad >= 2).sum()
        chk.append((ok if prior >= len(df_chunks) * 0.3 else warn,
                    f"Fragmentos con ≥2 pistas del dominio (prioritarios para la Persona 3): {prior}/{len(df_chunks)}"))
    idiomas = df_docs.idioma.value_counts().to_dict() if len(df_docs) else {}
    lineas = ["# Reporte de extracción de texto (Persona 2 · Glaciares)", "", f"Generado: {ahora()} · versión {VERSION}",
              "", f"OCR: {ocr_msg}", "", "| Estado | Verificación |", "|---|---|"] + [f"| {e} | {m} |" for e, m in chk]
    lineas += ["", "## Corpus procesado", "",
               f"- Páginas: {int(manifiesto.n_pages.fillna(0).sum())} · caracteres útiles: {int(manifiesto.n_chars_util.fillna(0).sum()):,}",
               f"- Idioma: {idiomas}",
               "- Por fuente: " + (", ".join(f"{k}: {v}" for k, v in df_docs.fuente.value_counts().items()) if len(df_docs) else ""),
               "- Documentos con cada sección: " + ", ".join(f"{k}: {v}" for k, v in cobertura.items())]
    if len(df_chunks):
        conteo_p = Counter(p for ps in df_chunks.pistas for p in ps)
        lineas += ["", "## Fragmentos (chunks)", "",
                   f"- Total: {len(df_chunks)} · promedio por documento: {len(df_chunks) / max(n_docs, 1):.1f}",
                   f"- Tamaño medio: {df_chunks.n_chars.mean():.0f} caracteres",
                   "- Por tipo de sección: " + ", ".join(f"{k}: {v}" for k, v in df_chunks.section_type.value_counts().items()),
                   "- Por origen del texto: " + ", ".join(f"{k}: {v}" for k, v in df_chunks.fuente_texto.value_counts().items()),
                   "- Fragmentos con cada pista: " + ", ".join(f"{k}: {v}" for k, v in conteo_p.most_common())]
    lineas += ["", "## Otros", "", f"- Leyendas (figuras, fotos, cuadros, mapas): {len(leyendas)}",
               f"- Tablas extraídas: {len(tablas)}", f"- Secciones totales: {len(df_sec)}"]
    problemas = manifiesto[flags != ""][["paper_id", "flags"]]
    if len(problemas):
        lineas += ["", "## Documentos con observaciones", "", "| paper_id | observaciones |", "|---|---|"]
        lineas += [f"| {r.paper_id} | {r.flags} |" for r in problemas.itertuples()]
    return chk, "\n".join(lineas), cobertura


def plantilla_anotacion(papers, n=12, semilla=7):
    """Muestra estratificada por fuente para anotar a mano (gold standard).
    Sirve para medir la extracción de la Persona 3 y adelanta la evaluación de la semana 12."""
    p = papers.copy()
    partes = [g.sample(min(len(g), max(1, n // p["fuente"].nunique())), random_state=semilla) for _, g in p.groupby("fuente")]
    m = pd.concat(partes)
    if len(m) < n:
        m = pd.concat([m, p[~p.paper_id.isin(m.paper_id)].sample(min(n - len(m), len(p) - len(m)), random_state=semilla)])
    cols = ["lagunas", "glaciares_nevados", "cordillera_cuenca_departamento", "nivel_de_riesgo", "eventos_y_fechas",
            "centros_poblados_expuestos", "obras_de_seguridad", "metodos_de_monitoreo", "instituciones",
            "mediciones_clave", "anotador", "notas"]
    salida = m[["paper_id", "title", "fuente"]].head(n).copy()
    for c in cols:
        salida[c] = ""
    return salida


README_TEXTO = """# Texto procesado del corpus Glaciares (Persona 2)

Entrada: `corpus_glaciares/papers.csv`, `pdfs/` y `pdfs_manifest.csv` (Persona 1).
Herramienta: PyMuPDF (tipografía y posición) + Tesseract para páginas escaneadas (opcional).

## Archivos

| Archivo | Contenido | Lo usa |
|---|---|---|
| `documentos_procesados.csv` | Una fila por documento: fuente, idioma, páginas, origen del texto (pdf, pdf+ocr, metadatos), secciones, fragmentos | Entrega del hito, control |
| `textos/<id>.txt` | Texto limpio completo de cada documento, con sus secciones | Lectura, revisión |
| `chunks.jsonl` (`.gz`) | Fragmentos de ~1500 caracteres con procedencia (documento, sección, páginas) y pistas del dominio | **Persona 3 (entidades y relaciones)** y RAG (semana 9) |
| `sections.csv` | Una fila por sección: tipo normalizado, título, páginas, texto | Persona 3, análisis |
| `tables.jsonl` | Tablas en Markdown (inventarios, áreas, volúmenes, cotas) | Persona 3 (valores numéricos) |
| `captions.csv` | Leyendas de figuras, fotos, cuadros y mapas | Persona 3 |
| `first_pages.jsonl` | Portada: instituciones candidatas, fechas | Persona 3 (publicadoPor, fechas) |
| `texto_manifest.csv` | Estado por PDF, hash, páginas escaneadas y con OCR, observaciones | Control de calidad |
| `reporte_extraccion.md` | Verificaciones automáticas | Entrega del hito |
| `anotacion_manual_plantilla.csv` | 12 documentos para anotar a mano (gold standard) | Evaluación de la extracción |

## Tipos de sección

`resumen, presentacion, introduccion, objetivos, antecedentes, area_estudio, metodologia, resultados,
desarrollo, discusion, conclusiones, recomendaciones, referencias, anexos, agradecimientos, indice,
front_matter, cuerpo, otro`

- `desarrollo`: capítulos propios del informe entre la introducción y las conclusiones
  (p. ej. "Descripción de los peligros", "Laguna Palcacocha").
- `cuerpo`: documento sin estructura detectable; se fragmenta completo.
- El índice, las referencias y los agradecimientos no se fragmentan.

## Campos de `chunks.jsonl`

`chunk_id` (`<documento>::s<sección>::c<n>`), `paper_id`, `fuente`, `title`, `year`, `section_idx`,
`section_type`, `heading`, `page_start`, `page_end`, `text`, `n_chars`, `pistas`, `prioridad`, `fuente_texto`.

- `pistas`: qué menciona el fragmento: `laguna`, `glaciar`, `peligro`, `poblado`, `obra`, `ubicacion`, `medicion`.
- `prioridad`: número de pistas (medición cuenta 0.5). Los fragmentos con prioridad ≥ 2 concentran los hechos
  que responden las preguntas de competencia; la Persona 3 puede empezar por ellos.
- `fuente_texto`: `pdf`, `ocr` (página escaneada leída con Tesseract) o `metadatos` (solo título y resumen).

## Uso de los textos

Los documentos son publicaciones de acceso abierto de instituciones públicas peruanas (INAIGEM, ANA, SENAMHI).
Se cita siempre la fuente con `paper_id` (handle del repositorio) y las páginas del fragmento.
"""


print("Persona 2 (reparación) · versión", VERSION, "· PyMuPDF", pymupdf.VersionBind)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 66.9 MB/s eta 0:00:00
Persona 2 (reparación) · versión 2.0-glaciares · PyMuPDF 1.28.2


## 3. Documentos a reparar y diagnóstico

In [4]:
papers = pd.read_csv(os.path.join(CARPETA_CORPUS, "papers.csv"), dtype={"paper_id": str})
ruta_man_p1 = os.path.join(CARPETA_CORPUS, "pdfs_manifest.csv")
man_p1 = (pd.read_csv(ruta_man_p1, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
          if os.path.exists(ruta_man_p1) else None)
manifiesto = pd.read_csv(os.path.join(CARPETA_TEXTO, "texto_manifest.csv"), dtype={"paper_id": str})
print("Estado actual:", manifiesto.estado.value_counts().to_dict())

A_REPARAR = manifiesto[manifiesto.estado.isin(CONFIG["ESTADOS_A_REPARAR"])].paper_id.tolist()
print(f"Documentos a reparar: {len(A_REPARAR)}\n")

diag = []
for pid in A_REPARAR:
    d = diagnosticar_pdf(ruta_pdf_de(pid, CARPETA_PDFS, man_p1))
    fila = papers.set_index("paper_id").loc[pid]
    diag.append({"paper_id": pid, "fuente": fila.fuente, "year": fila.year, "title": str(fila.title)[:60], **d})
diag = pd.DataFrame(diag)
if len(diag):
    print("Causa probable:", diag.causa_probable.value_counts().to_dict())
    with pd.option_context("display.max_rows", 100, "display.max_colwidth", 60):
        display(diag[["paper_id", "fuente", "year", "paginas", "car_por_pagina", "cobertura_imagen",
                      "causa_probable", "title", "texto_muestra"]])

OCR_OK, OCR_IDIOMA, OCR_MSG = preparar_ocr(True, RUTA_TESSDATA if os.path.isdir(RUTA_TESSDATA) else "")
print(("✅ " if OCR_OK and "spa" in OCR_IDIOMA else "⚠️ ") + OCR_MSG)
if len(diag):
    paginas = int(diag.paginas.clip(upper=CONFIG["MAX_PAGINAS_OCR"]).sum())
    print(f"Páginas a leer con OCR (como máximo): {paginas} → aprox. {paginas * 4 / 60:.0f} min")

Estado actual: {'ok': 186, 'ok_ocr': 180, 'solo_metadatos': 32}
Documentos a reparar: 32

Causa probable: {'escaneado con poco texto (sello o marca) → OCR': 18, 'revisar a mano': 14}


,paper_id,fuente,year,paginas,car_por_pagina,cobertura_imagen,causa_probable,title,texto_muestra
0,20.500.12748/704,INAIGEM,2026,5,360,0.40,revisar a mano,MONITOREO FOTOGRÁFICO DEL GLACIAR COLGANTE EN EL GLACIAR...,MONITOREO FOTOGRÁFICO DEL GLACIAR COLGANTE EN EL GLACIAR...
1,20.500.12748/703,INAIGEM,2026,5,343,0.39,revisar a mano,MONITOREO FOTOGRÁFICO DEL GLACIAR COLGANTE EN EL GLACIAR...,MONITOREO FOTOGRÁFICO DEL GLACIAR COLGANTE EN EL GLACIAR...
2,20.500.12748/701,INAIGEM,2026,5,508,0.41,revisar a mano,MONITOREO FOTOGRÁFICO DEL GLACIAR COLGANTE EN EL GLACIAR...,MONITOREO FOTOGRÁFICO DEL GLACIAR COLGANTE EN EL GLACIAR...
3,20.500.12748/654,INAIGEM,2025,1,2032,0.07,revisar a mano,Bofedales: Esponjas de vida en la montaña,BOFEDALES E s p o n j a s d e v i d a e n l a m o n t a ...
4,20.500.12748/624,INAIGEM,2023,1,165,1.00,escaneado con poco texto (sello o marca) → OCR,Landslide hazard assessment and risk reduction in the ru...,Landslide hazard assessment and risk reduction in the ru...
5,20.500.12748/623,INAIGEM,2024,1,169,1.00,escaneado con poco texto (sello o marca) → OCR,Assessing Peru’s Land Monitoring System Contributions to...,Assessing Peru’s Land Monitoring System Contributions to...
6,20.500.12748/622,INAIGEM,2024,1,113,1.00,escaneado con poco texto (sello o marca) → OCR,Future changes of precipitation types in the Peruvian Andes,Future changes of precipitation types in the Peruvian An...
7,20.500.12748/621,INAIGEM,2024,1,177,1.00,escaneado con poco texto (sello o marca) → OCR,Atmospheric black carbon observations and its valley-mou...,Atmospheric black carbon observations and its valley-mou...
8,20.500.12748/571,INAIGEM,2024,5,824,0.72,escaneado con poco texto (sello o marca) → OCR,"Lagunas Glaciares con riesgo de desborde, Respuestas a p...",L A G U N A S G L A C I A R E S C O N R I E S G O D E D ...
9,20.500.12748/497,INAIGEM,2023,1,379,1.00,escaneado con poco texto (sello o marca) → OCR,Unravelling and understanding local perceptions of water...,Autor(es) Sally Rangecroft Rosa Maria Dextre Isabel Rich...


✅ OCR disponible (español) en /usr/share/tesseract-ocr/5/tessdata/
Páginas a leer con OCR (como máximo): 130 → aprox. 9 min


## 4. Reprocesar solo esos documentos
Muestra el resultado de cada uno (`antes → ahora`). Si se corta, volver a ejecutar desde la sección 3: los ya reparados ya no aparecen como `solo_metadatos`.

In [5]:
import time
t0 = time.time()
cambios, manifiesto = reprocesar_documentos(
    A_REPARAR, papers, CARPETA_PDFS, CARPETA_TEXTO, man_p1, ocr=(OCR_OK, OCR_IDIOMA),
    max_paginas_ocr=CONFIG["MAX_PAGINAS_OCR"], extraer_tablas=CONFIG["EXTRAER_TABLAS"],
    max_paginas_tablas=CONFIG["MAX_PAGINAS_TABLAS"])
print(f"\nTiempo: {(time.time() - t0) / 60:.1f} min")
if len(cambios):
    print("Resultado:", cambios.ahora.value_counts().to_dict())
    siguen = cambios[cambios.ahora == "solo_metadatos"]
    if len(siguen):
        print(f"\nSiguen sin texto: {len(siguen)} (mapas, láminas o PDFs sin texto ni imagen legible). "
              "Se quedan con título y resumen; no afecta al hito.")
        display(siguen.merge(diag[["paper_id", "causa_probable", "title"]], on="paper_id"))
print("\nEstado total:", manifiesto.estado.value_counts().to_dict())

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
   1/32 20.500.12748/704: solo_metadatos → ok_ocr (3 pág. OCR, 2,233 car.)
   2/32 20.500.12748/703: solo_metadatos → ok_ocr (3 pág. OCR, 2,376 car.)
   3/32 20.500.12748/701: solo_metadatos → ok_ocr (3 pág. OCR, 3,287 car.)
   4/32 20.500.12748/654: solo_metadatos → solo_metadatos (0 pág. OCR, 281 car.)
   5/32 20.500.12748/624: solo_metadatos → solo_metadatos (1 pág. OCR, 272 car.)
   6/32 20.500.12748/623: solo_metadatos → solo_metadatos (1 pág. OCR, 251 car.)
   7/32 20.500.12748/622: solo_metadatos → solo_metadatos (1 pág. OCR, 252 car.)
   8/32 20.500.12748/621: solo_metadatos → solo_metadatos (1 pág. OCR, 278 car.)
   9/32 20.500.12748/571: solo_metadatos → ok_ocr (3 pág. OCR, 3,474 car.)
   10/32 20.500.12748/497: solo_metadatos → solo_metadatos (1 pág. OCR, 492 car.)
   11/32 20.500.12748/493: solo_metadatos → solo_metadatos (1 pág. OCR, 351 car.)
   12/32 20.500.12748/463: solo_metadatos → 

,paper_id,antes,ahora,paginas_ocr,car_utiles,causa_probable,title
0,20.500.12748/654,solo_metadatos,solo_metadatos,0,281,revisar a mano,Bofedales: Esponjas de vida en la montaña
1,20.500.12748/624,solo_metadatos,solo_metadatos,1,272,escaneado con poco texto (sello o marca) → OCR,Landslide hazard assessment and risk reduction...
2,20.500.12748/623,solo_metadatos,solo_metadatos,1,251,escaneado con poco texto (sello o marca) → OCR,Assessing Peru’s Land Monitoring System Contri...
3,20.500.12748/622,solo_metadatos,solo_metadatos,1,252,escaneado con poco texto (sello o marca) → OCR,Future changes of precipitation types in the P...
4,20.500.12748/621,solo_metadatos,solo_metadatos,1,278,escaneado con poco texto (sello o marca) → OCR,Atmospheric black carbon observations and its ...
5,20.500.12748/497,solo_metadatos,solo_metadatos,1,492,escaneado con poco texto (sello o marca) → OCR,Unravelling and understanding local perception...
6,20.500.12748/493,solo_metadatos,solo_metadatos,1,351,escaneado con poco texto (sello o marca) → OCR,Characteristics of cloud properties over South...
7,20.500.12748/460,solo_metadatos,solo_metadatos,1,438,escaneado con poco texto (sello o marca) → OCR,Mapping the benefits and the exchange values o...
8,20.500.12748/423,solo_metadatos,solo_metadatos,1,348,escaneado con poco texto (sello o marca) → OCR,Payment for ecosystem services in Peru: Assess...
9,20.500.12748/420,solo_metadatos,solo_metadatos,1,376,escaneado con poco texto (sello o marca) → OCR,Spatiotemporal Changes in Land Use and Ecosyst...



Estado total: {'ok_ocr': 187, 'ok': 186, 'solo_metadatos': 25}


## 5. Regenerar las salidas con los 398 documentos

In [6]:
resultados = cargar_resultados(papers, CARPETA_TEXTO)
print(f"Resultados cargados desde json/: {len(resultados)}")
df_docs, df_sec, df_chunks, primeras, leyendas, tablas = construir_salidas(
    resultados, papers, CARPETA_TEXTO, CONFIG["CHUNK_OBJETIVO"], CONFIG["CHUNK_MAXIMO"], CONFIG["SOLAPE"])
checks, reporte, cobertura = reporte_extraccion(manifiesto, df_docs, df_sec, df_chunks, leyendas, tablas,
                                                len(papers), OCR_MSG)
with open(os.path.join(CARPETA_TEXTO, "reporte_extraccion.md"), "w", encoding="utf-8") as f:
    f.write(reporte)
with open(os.path.join(CARPETA_TEXTO, "README_texto.md"), "w", encoding="utf-8") as f:
    f.write(README_TEXTO)
for e, m in checks:
    print(e, m)
print(f"\nOrigen del texto: {df_docs.fuente_texto.value_counts().to_dict()}")
print(f"Fragmentos: {len(df_chunks)} | prioritarios (≥2 pistas): {(df_chunks.prioridad >= 2).sum()} | "
      f"tablas: {len(tablas)} | leyendas: {len(leyendas)}")

Resultados cargados desde json/: 398
✅ Documentos con texto del PDF: 373/398 (de ellos con OCR: 187)
⚠️ Documentos solo con título + resumen (escaneados sin OCR): 25
✅ Documentos sin PDF o con PDF dañado: 0
✅ Documentos sin ningún fragmento: 0
✅ Documentos sin secciones detectadas (se fragmenta el cuerpo completo): 52
✅ Fragmentos con ≥2 pistas del dominio (prioritarios para la Persona 3): 9635/27416

Origen del texto: {'pdf+ocr': 187, 'pdf': 186, 'metadatos': 25}
Fragmentos: 27416 | prioritarios (≥2 pistas): 9635 | tablas: 4195 | leyendas: 8473


## Después
- Volver a subir a GitHub (`data/persona2/`) los archivos regenerados: `documentos_procesados.csv`, `chunks.jsonl.gz`, `sections.csv`, `tables.jsonl`, `captions.csv`, `first_pages.jsonl`, `texto_manifest.csv`, `reporte_extraccion.md`, y este notebook.
- La Persona 3 debe usar el `chunks.jsonl` **nuevo**.